In [ ]:
# pickle file inspector for influencer_22 dataset
import pickle
import numpy as np
import pandas as pd

# ===================== SET YOUR PATH HERE =====================
PKL_PATH = "/4tb/dataset/social_media/influencer_22/pickles/akshaykumar.pkl"
# ==============================================================

def describe_value(val, indent=0, max_list_items=5):
    pad = "  " * indent
    t = type(val).__name__

    if isinstance(val, dict):
        print(f"{pad}dict ({len(val)} keys)")
        for k, v in val.items():  # ← ALL keys, no limit
            print(f"{pad}  [{repr(k)}]:")
            describe_value(v, indent + 2, max_list_items)

    elif isinstance(val, (list, tuple)):
        cname = "list" if isinstance(val, list) else "tuple"
        print(f"{pad}{cname} (len={len(val)})")
        for i, item in enumerate(val[:max_list_items]):
            print(f"{pad}  [{i}]:")
            describe_value(item, indent + 2, max_list_items)
        if len(val) > max_list_items:
            print(f"{pad}  ... ({len(val) - max_list_items} more items)")

    elif isinstance(val, np.ndarray):
        print(f"{pad}np.ndarray | shape={val.shape} | dtype={val.dtype} | min={val.min():.4g} | max={val.max():.4g}")

    elif isinstance(val, pd.DataFrame):
        print(f"{pad}pd.DataFrame | shape={val.shape} | columns={list(val.columns)}")
        print(val.head(3).to_string(index=True).replace("\n", f"\n{pad}  "))

    elif isinstance(val, pd.Series):
        print(f"{pad}pd.Series | len={len(val)} | dtype={val.dtype} | name={val.name}")

    elif isinstance(val, str):
        preview = val[:80] + ("..." if len(val) > 80 else "")
        print(f"{pad}str (len={len(val)}): {repr(preview)}")

    elif isinstance(val, (int, float, bool, np.integer, np.floating)):
        print(f"{pad}{t}: {val}")

    elif val is None:
        print(f"{pad}None")

    else:
        print(f"{pad}{t}: {repr(val)[:120]}")


# ── Load & print ──────────────────────────────────────────────
with open(PKL_PATH, "rb") as f:
    data = pickle.load(f)

print("=" * 60)
print(f"FILE : {PKL_PATH}")
print(f"TYPE : {type(data).__name__}")
print("=" * 60)

# Since it's a list of posts, just inspect the first item fully
# to see the complete schema, then summarise the rest
print("\n── SCHEMA (first item) ──────────────────────────────────")
describe_value(data[0])

print(f"\n── SUMMARY ({len(data)} total items) ───────────────────────")
all_keys = set()
for item in data:
    if isinstance(item, dict):
        all_keys.update(item.keys())
print(f"All unique keys across all records: {sorted(all_keys)}")

FILE : /4tb/dataset/social_media/influencer_22/pickles/akshaykumar.pkl
TYPE : list

── SCHEMA (first item) ──────────────────────────────────
dict (17 keys)
  ['platformId']:
    str (len=29): '2992179251260570069_907025384'
  ['platform']:
    str (len=9): 'Instagram'
  ['date']:
    str (len=19): '2022-12-13 07:13:04'
  ['updated']:
    str (len=19): '2023-02-22 05:28:04'
  ['type']:
    str (len=5): 'photo'
  ['description']:
    str (len=433): 'My mantra for today - Garmi, humidity aur faux fur…Sab chalega, bas kaam kar, ka...'
  ['postUrl']:
    str (len=40): 'https://www.instagram.com/p/CmGWyz_rjnV/'
  ['subscriberCount']:
    int: 63610982
  ['score']:
    float: 1.5201705023571264
  ['media']:
    list (len=1)
      [0]:
        dict (4 keys)
          ['type']:
            str (len=5): 'photo'
          ['url']:
            str (len=274): 'https://scontent-sea1-1.cdninstagram.com/v/t51.29350-15/319669202_14637707484822...'
          ['height']:
            int: 1247
          

In [ ]:
# 500 random samples from the dataset for inspection
import os
import random
import shutil
import json
from pathlib import Path
from collections import Counter

# ── 1. SUMMARY ────────────────────────────────────────────────────────────────
SOURCE_DIR = "/4tb/dataset/social_media/influencer_22"
DEST_DIR   = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500"
MAP_DIR    = "/home/labad/minxing/code/Qwen3-VL/data_influencer22"
N_SAMPLE   = 500
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

print("=" * 60)
print(f"SOURCE : {SOURCE_DIR}")
print("=" * 60)
print("\n📂 Scanning all files recursively with pathlib.Path.rglob()...")

all_files = list(Path(SOURCE_DIR).rglob("*"))
all_files = [f for f in all_files if f.is_file()]

ext_counts = Counter(f.suffix.lower() if f.suffix else "(no extension)" for f in all_files)
print(f"   Total files found : {len(all_files)}")
print("\n📊 File type breakdown (sorted by count):")
for ext, count in ext_counts.most_common():
    tag = " ← IMAGE" if ext.lower() in IMAGE_EXTS else ""
    print(f"   {count:>6}  {ext}{tag}")

image_files = [f for f in all_files if f.suffix.lower() in IMAGE_EXTS]
print(f"\n🖼️  Total image files : {len(image_files)}")
print(f"   Image types       : {IMAGE_EXTS}")
print(f"   Method used       : pathlib.rglob() + suffix filtering + collections.Counter")

# ── 2. SAMPLE & COPY ──────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"🎲 Randomly sampling {N_SAMPLE} images (random.sample, fixed seed=42)...")

random.seed(42)
sampled = random.sample(image_files, N_SAMPLE)

os.makedirs(DEST_DIR, exist_ok=True)
print(f"📁 Destination folder created: {DEST_DIR}")

mapping = {}  # new_path -> original_path

print(f"\n📋 Copying and renaming files 1–{N_SAMPLE}...")
for i, src in enumerate(sampled, start=1):
    new_name = f"{i}{src.suffix.lower()}"
    dst = Path(DEST_DIR) / new_name
    shutil.copy2(src, dst)
    mapping[str(dst)] = str(src)

print(f"   ✅ Done. {N_SAMPLE} files copied to {DEST_DIR}")

# ── 3. SAVE MAPPING ───────────────────────────────────────────────────────────
map_path = Path(MAP_DIR) / "sample_500_mapping.json"
with open(map_path, "w") as f:
    json.dump(mapping, f, indent=2)

print(f"\n💾 Mapping saved to: {map_path}")
print(f"   Format  : JSON  (new_path → original_path)")
print(f"   Entries : {len(mapping)}")
print(f"\n📌 Example entries:")
for new, orig in list(mapping.items())[:3]:
    print(f"   {new}")
    print(f"   └─ {orig}")

SOURCE : /4tb/dataset/social_media/influencer_22

📂 Scanning all files recursively with pathlib.Path.rglob()...
   Total files found : 20173

📊 File type breakdown (sorted by count):
    18601  .jpg ← IMAGE
     1206  .webp ← IMAGE
      243  (no extension)
      102  .pkl
       18  .png ← IMAGE
        2  .zip
        1  .txt

🖼️  Total image files : 19825
   Image types       : {'.webp', '.png', '.jpg', '.jpeg'}
   Method used       : pathlib.rglob() + suffix filtering + collections.Counter

🎲 Randomly sampling 500 images (random.sample, fixed seed=42)...
📁 Destination folder created: /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500

📋 Copying and renaming files 1–500...
   ✅ Done. 500 files copied to /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500

💾 Mapping saved to: /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_mapping.json
   Format  : JSON  (new_path → original_path)
   Entries : 500

📌 Example entries:
   /home/labad/minxing/code/Q

In [ ]:
# ===================== CHECK SUBFOLDERS & ZIP FILES =====================
import os
from pathlib import Path
from collections import defaultdict

SOURCE_DIR = "/4tb/dataset/social_media/influencer_22"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# ── 1. CHECK SECOND-LAYER SUBFOLDERS (images/<platform>/<folder>) ─────────────
print("=" * 60)
print("📂 Checking second-layer subfolders under images/")
print("=" * 60)

images_root = Path(SOURCE_DIR) / "images"

violation_folders = []   # folders with more than 1 file
empty_folders     = []   # folders with 0 files
ok_folders        = 0

for platform_dir in sorted(images_root.iterdir()):
    if not platform_dir.is_dir():
        continue
    for leaf_dir in sorted(platform_dir.iterdir()):
        if not leaf_dir.is_dir():
            continue
        files = [f for f in leaf_dir.iterdir() if f.is_file()]
        if len(files) == 0:
            empty_folders.append(leaf_dir)
        elif len(files) > 1:
            violation_folders.append((leaf_dir, files))
        else:
            ok_folders += 1

print(f"\n✅ Folders with exactly 1 file : {ok_folders}")
print(f"⚠️  Empty folders               : {len(empty_folders)}")
print(f"❌ Folders with >1 file         : {len(violation_folders)}")

if violation_folders:
    print(f"\n🔍 Violation details (first 20):")
    for folder, files in violation_folders[:20]:
        print(f"\n   📁 {folder}  ({len(files)} files)")
        for f in files:
            print(f"      └─ {f.name}")

if empty_folders:
    print(f"\n🔍 Empty folders (first 20):")
    for folder in empty_folders[:20]:
        print(f"   📁 {folder}")

# ── 2. FIND ALL ZIP FILES ──────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("🗜️  Scanning for zip files recursively...")
print("=" * 60)

ZIP_EXTS = {".zip", ".tar", ".gz", ".tar.gz", ".tgz", ".bz2", ".tar.bz2", ".rar", ".7z"}

zip_files = [
    f for f in Path(SOURCE_DIR).rglob("*")
    if f.is_file() and f.suffix.lower() in ZIP_EXTS
]

print(f"\n📦 Total zip/archive files found: {len(zip_files)}")

# Group by parent directory
by_dir = defaultdict(list)
for z in zip_files:
    by_dir[str(z.parent)].append(z)

print(f"   Spread across {len(by_dir)} directories\n")

for parent, files in sorted(by_dir.items()):
    print(f"📁 {parent}")
    for f in files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"   └─ {f.name}  ({size_mb:.1f} MB)")

📂 Checking second-layer subfolders under images/

✅ Folders with exactly 1 file : 18928
⚠️  Empty folders               : 0
❌ Folders with >1 file         : 307

🔍 Violation details (first 20):

   📁 /4tb/dataset/social_media/influencer_22/images/fcbarcelona/Cl-985XgIY9  (3 files)
      └─ 3.webp
      └─ 2.jpg
      └─ 1.jpg

   📁 /4tb/dataset/social_media/influencer_22/images/fcbarcelona/CmJv9pTKxu9  (5 files)
      └─ 5.webp
      └─ 3.jpg
      └─ 4.jpg
      └─ 2.jpg
      └─ 1.jpg

   📁 /4tb/dataset/social_media/influencer_22/images/juventus/Cf1BH5fqr5X  (3 files)
      └─ 3.jpg
      └─ 2.jpg
      └─ 1.jpg

   📁 /4tb/dataset/social_media/influencer_22/images/juventus/Cf33UudKQd_  (3 files)
      └─ 3.jpg
      └─ 2.jpg
      └─ 1.jpg

   📁 /4tb/dataset/social_media/influencer_22/images/juventus/Cf3g_XSo8o4  (3 files)
      └─ 1.webp
      └─ 3.jpg
      └─ 2.jpg

   📁 /4tb/dataset/social_media/influencer_22/images/juventus/Cf3n1t6oJKn  (3 files)
      └─ 1.webp
      └─ 3.jpg
 

In [ ]:
# ===================== FULL IMAGE SCAN & CORRUPTION ANALYSIS =====================
import os
import json
import pickle
from pathlib import Path
from PIL import Image
import imghdr
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
from collections import defaultdict, Counter

SOURCE_DIR  = "/4tb/dataset/social_media/influencer_22"
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".webp"}
NUM_WORKERS = 8  # tune with: nproc

# ── WORKER ────────────────────────────────────────────────────────────────────
def check_image(img_path_str):
    img_path = Path(img_path_str)
    result = {
        "path"         : img_path_str,
        "status"       : "ok",
        "failure_stage": None,
        "error_type"   : None,
        "error_msg"    : None,
        "wrong_ext"    : False,
        "declared_ext" : img_path.suffix.lower().strip("."),
        "detected_fmt" : None,
        "file_size_kb" : None,
        "platform"     : img_path.parts[-3] if len(img_path.parts) >= 3 else "unknown",
        "leaf_folder"  : str(img_path.parent),
    }

    try:
        size = img_path.stat().st_size
        result["file_size_kb"] = round(size / 1024, 2)
    except OSError as e:
        result["status"]        = "unreadable"
        result["failure_stage"] = "os_stat"
        result["error_type"]    = type(e).__name__
        result["error_msg"]     = str(e)
        return result

    if size == 0:
        result["status"] = "zero_byte"
        return result

    try:
        with open(img_path, "rb") as f:
            f.read(16)
    except OSError as e:
        result["status"]        = "unreadable"
        result["failure_stage"] = "os_read"
        result["error_type"]    = type(e).__name__
        result["error_msg"]     = str(e)
        return result

    try:
        detected = imghdr.what(img_path)
        result["detected_fmt"] = detected
        declared = result["declared_ext"]
        declared_norm = "jpeg" if declared == "jpg" else declared
        if detected and detected != declared_norm:
            result["wrong_ext"] = True
    except Exception:
        pass

    try:
        with Image.open(img_path) as im:
            im.verify()
    except Exception as e:
        result["status"]        = "corrupted"
        result["failure_stage"] = "verify"
        result["error_type"]    = type(e).__name__
        result["error_msg"]     = str(e)
        return result

    try:
        with Image.open(img_path) as im:
            im.load()
    except Exception as e:
        result["status"]        = "corrupted"
        result["failure_stage"] = "load"
        result["error_type"]    = type(e).__name__
        result["error_msg"]     = str(e)
        return result

    return result


# ── PICKLE SCANNER ────────────────────────────────────────────────────────────
def scan_all_pickles(source_dir):
    pickle_dir = Path(source_dir) / "pickles"
    pkl_files  = list(pickle_dir.rglob("*.pkl")) if pickle_dir.exists() else []
    print(f"\n🥒 Found {len(pkl_files):,} pickle files in {pickle_dir}")

    shortcode_to_type     = {}
    type_counter          = Counter()
    unknown_type_examples = []

    for pkl_path in tqdm(pkl_files, desc="📦 Reading pickles", unit="pkl"):
        try:
            with open(pkl_path, "rb") as f:
                data = pickle.load(f)
        except Exception as e:
            print(f"   ⚠️  Failed to load {pkl_path.name}: {e}")
            continue

        posts = data if isinstance(data, list) else [data]
        for post in posts:
            if not isinstance(post, dict):
                continue

            # ── correct type key ──────────────────────────────────────
            post_type = post.get("type", "unknown").strip().lower()
            type_counter[post_type] += 1

            # ── extract shortcode from postUrl ────────────────────────
            shortcode = None
            post_url  = post.get("postUrl", "")
            if "/p/" in post_url:
                shortcode = post_url.rstrip("/").split("/p/")[-1].split("/")[0]

            # fallback chain if no postUrl
            if not shortcode:
                shortcode = (
                    post.get("platformId") or
                    post.get("legacyId")   or
                    str(post.get("id", ""))
                ) or None

            if shortcode:
                shortcode_to_type[str(shortcode)] = post_type

            if post_type == "unknown":
                unknown_type_examples.append({
                    "pkl" : pkl_path.name,
                    "keys": list(post.keys())[:15],
                })

    print(f"\n📋 Post types found across all pickles:")
    for ptype, count in type_counter.most_common():
        print(f"   {count:>8,}  {ptype}")

    return shortcode_to_type, type_counter, unknown_type_examples


def cross_reference_type_vs_files(shortcode_to_type, folder_file_counts, source_dir):
    """
    For each post type, look at how many files are in its image folder.
    folder_file_counts: Counter(folder_path -> n_files)   (ALL files, not just images)
    """
    images_root = Path(source_dir) / "images"

    # build a map: leaf_folder_path -> file_count  (already have this)
    # but we need to match shortcode -> folder
    # structure: images/<platform>/<shortcode>/

    type_file_dist    = defaultdict(Counter)   # post_type -> Counter(n_files -> n_folders)
    type_missing_folder = Counter()            # post_type -> how many had no folder on disk
    type_found_folder   = Counter()

    for shortcode, post_type in shortcode_to_type.items():
        # search across all platforms
        found = False
        for platform_dir in images_root.iterdir():
            if not platform_dir.is_dir():
                continue
            candidate = platform_dir / shortcode
            if candidate.exists() and candidate.is_dir():
                n_files = folder_file_counts.get(str(candidate), 0)
                type_file_dist[post_type][n_files] += 1
                type_found_folder[post_type] += 1
                found = True
                break
        if not found:
            type_missing_folder[post_type] += 1

    return type_file_dist, type_found_folder, type_missing_folder


# ── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print(f"🔍 Scanning: {SOURCE_DIR}")
    print("=" * 60)

    # ── Step 1: collect image files ───────────────────────────────────────
    print("\n📂 Collecting image files...")
    image_files = [
        str(f) for f in Path(SOURCE_DIR).rglob("*")
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS
    ]
    print(f"🖼️  Total images found: {len(image_files):,}\n")

    # ── Step 2: folder file counts (ALL files, not just images) ──────────
    print("📂 Counting files per leaf folder...")
    folder_file_counts = Counter()
    for f in Path(SOURCE_DIR).rglob("*"):
        if f.is_file():
            folder_file_counts[str(f.parent)] += 1
    print(f"   {len(folder_file_counts):,} unique leaf folders found\n")

    # ── Step 3: scan all pickles ──────────────────────────────────────────
    shortcode_to_type, type_counter, unknown_type_examples = scan_all_pickles(SOURCE_DIR)

    # ── Step 4: cross-reference post type vs folder file counts ──────────
    print("\n📊 Cross-referencing post types with image folders...")
    type_file_dist, type_found_folder, type_missing_folder = cross_reference_type_vs_files(
        shortcode_to_type, folder_file_counts, SOURCE_DIR
    )

    print(f"\n📂 File-count distribution per post type:")
    for post_type, dist in sorted(type_file_dist.items()):
        total_folders = type_found_folder[post_type]
        missing       = type_missing_folder[post_type]
        print(f"\n   ── {post_type.upper()}  ({total_folders:,} folders found, {missing:,} missing on disk)")
        for n_files, n_folders in sorted(dist.items()):
            bar = "█" * min(40, int(40 * n_folders / max(total_folders, 1)))
            print(f"      {n_files:>3} file(s) in folder : {n_folders:>6,}  {bar}")

    # ── Step 5: run parallel image scan ──────────────────────────────────
    all_results = []
    print(f"\n⚙️  Running image validation with {NUM_WORKERS} workers...\n")
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(check_image, p): p for p in image_files}
        for future in tqdm(as_completed(futures), total=len(image_files), unit="img"):
            all_results.append(future.result())

    # ── Step 6: categorize ────────────────────────────────────────────────
    ok         = [r for r in all_results if r["status"] == "ok"]
    corrupted  = [r for r in all_results if r["status"] == "corrupted"]
    zero_byte  = [r for r in all_results if r["status"] == "zero_byte"]
    unreadable = [r for r in all_results if r["status"] == "unreadable"]
    wrong_ext  = [r for r in all_results if r["wrong_ext"]]
    bad        = corrupted + zero_byte + unreadable

    # ── Step 7: enrich bad results with post_type ─────────────────────────
    # leaf_folder name = shortcode
    for r in all_results:
        shortcode = Path(r["leaf_folder"]).name
        r["post_type"] = shortcode_to_type.get(shortcode, "unknown")

    # ── Step 8: bad files broken down by post type ────────────────────────
    bad_by_post_type = Counter(r["post_type"] for r in bad)
    ok_by_post_type  = Counter(r["post_type"] for r in ok)

    print(f"\n{'=' * 60}  SUMMARY")
    print(f"✅ OK               : {len(ok):,}")
    print(f"❌ Corrupted        : {len(corrupted):,}")
    print(f"💀 Zero-byte        : {len(zero_byte):,}")
    print(f"🚫 Unreadable (OS)  : {len(unreadable):,}")
    print(f"🔀 Wrong extension  : {len(wrong_ext):,}")
    print(f"{'=' * 60}")

    stage_counts = Counter(r["failure_stage"] for r in corrupted)
    print(f"\n🔬 Corruption stage breakdown:")
    for stage, count in stage_counts.most_common():
        label = {
            "verify": "header corrupt (verify failed)",
            "load"  : "body truncated (header OK, pixel decode failed)",
        }.get(stage, stage)
        print(f"   {count:>6}  {label}")

    error_type_counts = Counter(r["error_type"] for r in corrupted)
    print(f"\n🐛 Error types in corrupted files:")
    for etype, count in error_type_counts.most_common():
        print(f"   {count:>6}  {etype}")

    print(f"\n📱 Bad files by platform:")
    platform_bad = Counter(r["platform"] for r in bad)
    for platform, count in platform_bad.most_common():
        print(f"   {count:>6}  {platform}")

    print(f"\n🏷️  Bad files by post type:")
    for ptype, count in bad_by_post_type.most_common():
        total = bad_by_post_type[ptype] + ok_by_post_type[ptype]
        pct   = 100 * count / max(total, 1)
        print(f"   {count:>6,}  {ptype:<20}  ({pct:.1f}% of that type's images are bad)")

    print(f"\n📊 Distribution: bad files in single vs multi-file folders?")
    single_bad = [r for r in bad if folder_file_counts.get(r["leaf_folder"], 1) == 1]
    multi_bad  = [r for r in bad if folder_file_counts.get(r["leaf_folder"], 1) >  1]
    print(f"   Bad in single-file folders : {len(single_bad):,}")
    print(f"   Bad in multi-file folders  : {len(multi_bad):,}")

    multi_folder_counts = Counter(r["leaf_folder"] for r in multi_bad)
    print(f"\n   Top 10 multi-file folders with most bad images:")
    for folder, count in multi_folder_counts.most_common(10):
        total_in_folder = folder_file_counts[folder]
        shortcode       = Path(folder).name
        ptype           = shortcode_to_type.get(shortcode, "unknown")
        print(f"   {count} bad / {total_in_folder} total  [{ptype}]  →  {folder}")

    print(f"\n📊 Multi-file folder analysis:")
    multi_file_folders  = {f: c for f, c in folder_file_counts.items() if c > 1}
    single_file_folders = {f: c for f, c in folder_file_counts.items() if c == 1}
    print(f"   Total folders                    : {len(folder_file_counts):,}")
    print(f"   Single-file folders              : {len(single_file_folders):,}")
    print(f"   Multi-file folders               : {len(multi_file_folders):,}")

    multi_with_bad = set(r["leaf_folder"] for r in bad if r["leaf_folder"] in multi_file_folders)
    multi_all_good = set(multi_file_folders.keys()) - multi_with_bad
    print(f"\n   Of {len(multi_file_folders):,} multi-file folders:")
    print(f"   ❌ Has ≥1 bad image              : {len(multi_with_bad):,}")
    print(f"   ✅ All images fine               : {len(multi_all_good):,}")
    print(f"   📈 % multi-folders with bad      : {100*len(multi_with_bad)/max(len(multi_file_folders),1):.1f}%")

    single_with_bad = set(r["leaf_folder"] for r in bad if r["leaf_folder"] in single_file_folders)
    single_all_good = set(single_file_folders.keys()) - single_with_bad
    print(f"\n   Of {len(single_file_folders):,} single-file folders:")
    print(f"   ❌ Has bad image                 : {len(single_with_bad):,}")
    print(f"   ✅ Image is fine                 : {len(single_all_good):,}")
    print(f"   📈 % single-folders with bad     : {100*len(single_with_bad)/max(len(single_file_folders),1):.1f}%")

    # ── Step 9: save full JSON report ─────────────────────────────────────
    report = {
        "source"        : SOURCE_DIR,
        "total_scanned" : len(image_files),
        "summary": {
            "ok"        : len(ok),
            "corrupted" : len(corrupted),
            "zero_byte" : len(zero_byte),
            "unreadable": len(unreadable),
            "wrong_ext" : len(wrong_ext),
        },
        "corruption_stages"     : dict(stage_counts),
        "corruption_error_types": dict(error_type_counts),
        "bad_by_platform"       : dict(platform_bad),
        "post_type_counts"      : dict(type_counter),
        "bad_by_post_type"      : dict(bad_by_post_type),
        "ok_by_post_type"       : dict(ok_by_post_type),
        "post_type_folder_file_distribution": {
            ptype: {
                "found_on_disk" : type_found_folder[ptype],
                "missing_on_disk": type_missing_folder[ptype],
                "file_count_dist": {str(k): v for k, v in sorted(dist.items())},
            }
            for ptype, dist in type_file_dist.items()
        },
        "distribution": {
            "bad_in_single_file_folders": len(single_bad),
            "bad_in_multi_file_folders" : len(multi_bad),
        },
        "multi_file_folder_analysis": {
            "total_folders"                 : len(folder_file_counts),
            "single_file_folders"           : len(single_file_folders),
            "multi_file_folders"            : len(multi_file_folders),
            "multi_folders_with_bad_images" : len(multi_with_bad),
            "multi_folders_all_good"        : len(multi_all_good),
            "pct_multi_folders_with_bad"    : round(100*len(multi_with_bad)/max(len(multi_file_folders),1), 2),
            "single_folders_with_bad_images": len(single_with_bad),
            "single_folders_all_good"       : len(single_all_good),
            "pct_single_folders_with_bad"   : round(100*len(single_with_bad)/max(len(single_file_folders),1), 2),
        },
        "bad_files": [
            {k: v for k, v in r.items() if r["status"] != "ok"}
            for r in bad
        ],
        "wrong_ext_files": [
            {"path": r["path"], "declared": r["declared_ext"], "actual": r["detected_fmt"]}
            for r in wrong_ext
        ],
        "unknown_type_examples": unknown_type_examples[:10],
    }

    report_path = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/full_scan_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n💾 Full report saved to: {report_path}")

🔍 Scanning: /4tb/dataset/social_media/influencer_22

📂 Collecting image files...
🖼️  Total images found: 19,825

📂 Counting files per leaf folder...
   19,339 unique leaf folders found


🥒 Found 102 pickle files in /4tb/dataset/social_media/influencer_22/pickles


📦 Reading pickles: 100%|██████████| 102/102 [00:04<00:00, 23.04pkl/s]



📋 Post types found across all pickles:
     19,235  photo
     18,047  album
      7,810  video

📊 Cross-referencing post types with image folders...

📂 File-count distribution per post type:

⚙️  Running image validation with 8 workers...



100%|██████████| 19825/19825 [00:26<00:00, 760.81img/s] 



============================================================  SUMMARY
✅ OK               : 19,094
❌ Corrupted        : 730
💀 Zero-byte        : 1
🚫 Unreadable (OS)  : 0
🔀 Wrong extension  : 908

🔬 Corruption stage breakdown:
      730  header corrupt (verify failed)

🐛 Error types in corrupted files:
      730  UnidentifiedImageError

📱 Bad files by platform:
      280  manchesterunited
      230  juventus
       68  victoriassecret
       38  psg
       34  realmadrid
       32  milliebobbybrown
       30  snoopdogg
        6  kimkardashian
        6  fcbarcelona
        4  premierleague
        2  katyperry
        1  ellendegeneres

🏷️  Bad files by post type:
      731  unknown               (3.7% of that type's images are bad)

📊 Distribution: bad files in single vs multi-file folders?
   Bad in single-file folders : 1
   Bad in multi-file folders  : 730

   Top 10 multi-file folders with most bad images:
   10 bad / 11 total  [unknown]  →  /4tb/dataset/social_media/influencer_22

In [1]:
import pickle
from pathlib import Path

pkl = list(Path("/4tb/dataset/social_media/influencer_22/pickles").rglob("*.pkl"))[0]
with open(pkl, "rb") as f:
    data = pickle.load(f)

post = data[0]
print("All keys:", list(post.keys()))

# print a few candidate ID fields
for k in post:
    v = post[k]
    if isinstance(v, str) and len(v) < 50:
        print(f"  {k}: {v}")

All keys: ['platformId', 'platform', 'date', 'updated', 'type', 'description', 'postUrl', 'subscriberCount', 'score', 'media', 'statistics', 'account', 'likeAndViewCountsDisabled', 'history', 'languageCode', 'legacyId', 'id']
  platformId: 2993159058827500915_371299822
  platform: Instagram
  date: 2022-12-14 15:39:46
  updated: 2023-02-11 07:41:26
  type: album
  description: 🇦🇷🙌🏽
  postUrl: https://www.instagram.com/p/CmJ1k45p91z/
  languageCode: es
  id: 535778|2993159058827500915


In [5]:
# 500 random samples from the dataset for inspection (only valid images)
import os
import random
import shutil
import json
from pathlib import Path
from collections import Counter
from PIL import Image
import imghdr

# ── CONFIG ────────────────────────────────────────────────────────────────────
SOURCE_DIR = "/4tb/dataset/social_media/influencer_22"
DEST_DIR   = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_clean"
MAP_DIR    = "/home/labad/minxing/code/Qwen3-VL/data_influencer22"
N_SAMPLE   = 500
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SEED       = 42

# ── HELPER: check if a single image is fine ───────────────────────────────────
def is_valid_image(img_path: Path) -> bool:
    # zero byte
    try:
        if img_path.stat().st_size == 0:
            return False
    except OSError:
        return False

    # wrong extension
    try:
        detected = imghdr.what(img_path)
        declared = img_path.suffix.lower().strip(".")
        declared = "jpeg" if declared == "jpg" else declared
        if detected and detected != declared:
            return False
    except Exception:
        return False

    # corrupted header or truncated body
    try:
        with Image.open(img_path) as im:
            im.verify()
        with Image.open(img_path) as im:
            im.load()
    except Exception:
        return False

    return True

# ── 1. COLLECT ALL IMAGE FILES ────────────────────────────────────────────────
print("=" * 60)
print(f"SOURCE : {SOURCE_DIR}")
print("=" * 60)
print("\n📂 Scanning all image files...")

all_image_files = [
    f for f in Path(SOURCE_DIR).rglob("*")
    if f.is_file() and f.suffix.lower() in IMAGE_EXTS
]
print(f"   Total image files found : {len(all_image_files):,}")

# ── 2. GROUP BY LEAF FOLDER ───────────────────────────────────────────────────
print("\n📁 Grouping by leaf folder...")
folder_to_images: dict[str, list[Path]] = {}
for f in all_image_files:
    folder_to_images.setdefault(str(f.parent), []).append(f)

print(f"   Total leaf folders      : {len(folder_to_images):,}")

# ── 3. VALIDATE — keep only folders that have ≥1 valid image ─────────────────
print("\n🔍 Validating images (this may take a while)...")

valid_folders: dict[str, list[Path]] = {}  # folder -> list of valid images in it
total_checked  = 0
total_valid    = 0
total_invalid  = 0

for folder, images in folder_to_images.items():
    valid_in_folder = []
    for img in images:
        total_checked += 1
        if is_valid_image(img):
            valid_in_folder.append(img)
            total_valid += 1
        else:
            total_invalid += 1
    if valid_in_folder:
        valid_folders[folder] = valid_in_folder

print(f"   Images checked          : {total_checked:,}")
print(f"   ✅ Valid images          : {total_valid:,}")
print(f"   ❌ Invalid images        : {total_invalid:,}")
print(f"   📁 Folders with ≥1 valid : {len(valid_folders):,}")

if len(valid_folders) < N_SAMPLE:
    raise ValueError(
        f"Not enough valid folders! Found {len(valid_folders):,}, need {N_SAMPLE}."
    )

# ── 4. SAMPLE 500 FOLDERS, PICK 1 IMAGE PER FOLDER ───────────────────────────
print(f"\n🎲 Sampling {N_SAMPLE} folders (seed={SEED})...")
random.seed(SEED)

sampled_folders = random.sample(list(valid_folders.keys()), N_SAMPLE)

sampled_images: list[tuple[Path, str]] = []  # (chosen image path, its folder)
for folder in sampled_folders:
    candidates = valid_folders[folder]
    chosen     = random.choice(candidates)   # 1 random valid image per folder
    sampled_images.append((chosen, folder))

print(f"   ✅ {N_SAMPLE} images selected (one per folder)")

# ── 5. COPY & RENAME ──────────────────────────────────────────────────────────
os.makedirs(DEST_DIR, exist_ok=True)
print(f"\n📋 Copying to: {DEST_DIR}")

mapping = {}  # new_path -> {original_path, folder, folder_file_count, valid_file_count}
for i, (src, folder) in enumerate(sampled_images, start=1):
    new_name = f"{i}{src.suffix.lower()}"
    dst      = Path(DEST_DIR) / new_name
    shutil.copy2(src, dst)
    mapping[str(dst)] = {
        "original_path"     : str(src),
        "source_folder"     : folder,
        "files_in_folder"   : len(folder_to_images[folder]),
        "valid_in_folder"   : len(valid_folders[folder]),
    }

print(f"   ✅ Done. {N_SAMPLE} files copied.")

# ── 6. SAVE MAPPING ───────────────────────────────────────────────────────────
map_path = Path(MAP_DIR) / "sample_500_clean_mapping.json"
with open(map_path, "w") as f:
    json.dump(mapping, f, indent=2)

print(f"\n💾 Mapping saved to : {map_path}")
print(f"   Entries           : {len(mapping)}")
print(f"\n📌 Example entries:")
for new, meta in list(mapping.items())[:3]:
    print(f"   {new}")
    print(f"   └─ original : {meta['original_path']}")
    print(f"   └─ folder   : {meta['source_folder']}")
    print(f"   └─ files in folder (total / valid) : {meta['files_in_folder']} / {meta['valid_in_folder']}")

SOURCE : /4tb/dataset/social_media/influencer_22

📂 Scanning all image files...
   Total image files found : 19,825

📁 Grouping by leaf folder...
   Total leaf folders      : 19,095

🔍 Validating images (this may take a while)...
   Images checked          : 19,825
   ✅ Valid images          : 18,186
   ❌ Invalid images        : 1,639
   📁 Folders with ≥1 valid : 18,186

🎲 Sampling 500 folders (seed=42)...
   ✅ 500 images selected (one per folder)

📋 Copying to: /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_clean
   ✅ Done. 500 files copied.

💾 Mapping saved to : /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_clean_mapping.json
   Entries           : 500

📌 Example entries:
   /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_clean/1.jpg
   └─ original : /4tb/dataset/social_media/influencer_22/images/433/CdLBuxfLZsG/2831364403011361542_253625977.jpg
   └─ folder   : /4tb/dataset/social_media/influencer_22/images/433/CdLBuxfLZsG
   └─ files 

In [7]:
# 200 random samples from the 500 for inspection
import os
import random
import shutil
import json
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
SOURCE_DIR = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_clean"
DEST_DIR   = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_200"
MAP_DIR    = "/home/labad/minxing/code/Qwen3-VL/data_influencer22"
N_SAMPLE   = 200
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# ── 1. LOAD sample_500 files ──────────────────────────────────────────────────
all_images = [f for f in Path(SOURCE_DIR).iterdir()
              if f.is_file() and f.suffix.lower() in IMAGE_EXTS]

print(f"🖼️  Found {len(all_images)} image files in sample_500")

# ── 2. SAMPLE 200 ─────────────────────────────────────────────────────────────
random.seed(99)
sampled = random.sample(all_images, N_SAMPLE)
print(f"🎲 Randomly selected {N_SAMPLE} files (random.sample, seed=99)")

# ── 3. COPY keeping original names ───────────────────────────────────────────
os.makedirs(DEST_DIR, exist_ok=True)
print(f"📁 Destination folder created: {DEST_DIR}\n")

for src in sampled:
    dst = Path(DEST_DIR) / src.name  # keep same filename e.g. 7.jpg
    shutil.copy2(src, dst)

print(f"✅ {N_SAMPLE} files copied to {DEST_DIR}")

# ── 4. SAVE MAPPING (new path → original /4tb path via sample_500 mapping) ───
# Load the existing sample_500 mapping to trace back to /4tb originals
map_500_path = Path(MAP_DIR) / "sample_500_clean_mapping.json"
with open(map_500_path, "r") as f:
    mapping_500 = json.load(f)

# Build a lookup: filename (e.g. "7.jpg") → original /4tb path
filename_to_original = {Path(k).name: v for k, v in mapping_500.items()}

mapping_200 = {}
for src in sampled:
    new_path      = str(Path(DEST_DIR) / src.name)
    original_path = filename_to_original.get(src.name, str(src))  # fallback to src
    mapping_200[new_path] = original_path

map_200_path = Path(MAP_DIR) / "sample_200_mapping.json"
with open(map_200_path, "w") as f:
    json.dump(mapping_200, f, indent=2)

print(f"\n💾 Mapping saved to : {map_200_path}")
print(f"   Entries          : {len(mapping_200)}")
print(f"\n📌 Example entries:")
for new, orig in list(mapping_200.items())[:3]:
    print(f"   {new}")
    print(f"   └─ {orig}")

🖼️  Found 500 image files in sample_500
🎲 Randomly selected 200 files (random.sample, seed=99)
📁 Destination folder created: /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_200

✅ 200 files copied to /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_200

💾 Mapping saved to : /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_200_mapping.json
   Entries          : 200

📌 Example entries:
   /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_200/5.jpg
   └─ {'original_path': '/4tb/dataset/social_media/influencer_22/images/psg/CctK4K_r0aF/2822960381992846981_232024162.jpg', 'source_folder': '/4tb/dataset/social_media/influencer_22/images/psg/CctK4K_r0aF', 'files_in_folder': 1, 'valid_in_folder': 1}
   /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_200/301.jpg
   └─ {'original_path': '/4tb/dataset/social_media/influencer_22/images/juventus/Cab0iqSqnc6/2782048275122517818_194317179.jpg', 'source_folder': '/4tb/dataset/social_media/influencer_2

In [8]:
# 30 random samples from the 500 for inspection
import os
import random
import shutil
import json
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
SOURCE_DIR = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_500_clean"
DEST_DIR   = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30"
MAP_DIR    = "/home/labad/minxing/code/Qwen3-VL/data_influencer22"
N_SAMPLE   = 30
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# ── 1. LOAD sample_500 files ──────────────────────────────────────────────────
all_images = [f for f in Path(SOURCE_DIR).iterdir()
              if f.is_file() and f.suffix.lower() in IMAGE_EXTS]

print(f"🖼️  Found {len(all_images)} image files in sample_500")

# ── 2. SAMPLE 30 ─────────────────────────────────────────────────────────────
random.seed(99)
sampled = random.sample(all_images, N_SAMPLE)
print(f"🎲 Randomly selected {N_SAMPLE} files (random.sample, seed=99)")

# ── 3. COPY keeping original names ───────────────────────────────────────────
os.makedirs(DEST_DIR, exist_ok=True)
print(f"📁 Destination folder created: {DEST_DIR}\n")

for src in sampled:
    dst = Path(DEST_DIR) / src.name  # keep same filename e.g. 7.jpg
    shutil.copy2(src, dst)

print(f"✅ {N_SAMPLE} files copied to {DEST_DIR}")

# ── 4. SAVE MAPPING (new path → original /4tb path via sample_500 mapping) ───
# Load the existing sample_500 mapping to trace back to /4tb originals
map_500_path = Path(MAP_DIR) / "sample_500_clean_mapping.json"
with open(map_500_path, "r") as f:
    mapping_500 = json.load(f)

# Build a lookup: filename (e.g. "7.jpg") → original /4tb path
filename_to_original = {Path(k).name: v for k, v in mapping_500.items()}

mapping_30 = {}
for src in sampled:
    new_path      = str(Path(DEST_DIR) / src.name)
    original_path = filename_to_original.get(src.name, str(src))  # fallback to src
    mapping_30[new_path] = original_path

map_30_path = Path(MAP_DIR) / "sample_30_mapping.json"
with open(map_30_path, "w") as f:
    json.dump(mapping_30, f, indent=2)

print(f"\n💾 Mapping saved to : {map_30_path}")
print(f"   Entries          : {len(mapping_30)}")
print(f"\n📌 Example entries:")
for new, orig in list(mapping_30.items())[:3]:
    print(f"   {new}")
    print(f"   └─ {orig}")

🖼️  Found 500 image files in sample_500
🎲 Randomly selected 30 files (random.sample, seed=99)
📁 Destination folder created: /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30

✅ 30 files copied to /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30

💾 Mapping saved to : /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30_mapping.json
   Entries          : 30

📌 Example entries:
   /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/5.jpg
   └─ {'original_path': '/4tb/dataset/social_media/influencer_22/images/psg/CctK4K_r0aF/2822960381992846981_232024162.jpg', 'source_folder': '/4tb/dataset/social_media/influencer_22/images/psg/CctK4K_r0aF', 'files_in_folder': 1, 'valid_in_folder': 1}
   /home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/301.jpg
   └─ {'original_path': '/4tb/dataset/social_media/influencer_22/images/juventus/Cab0iqSqnc6/2782048275122517818_194317179.jpg', 'source_folder': '/4tb/dataset/social_media/influencer_22/images